| 配置 | 值 | 影响到哪些指标 |
|---|---|---|
| `adv_estimator` | **grpo** | `critic/advantages`、`returns` 用组内归一化,无 critic 网络 |
| `use_kl_in_reward` | **False** | ⇒ `critic/rewards == critic/score`(KL 不进 reward) |
| `use_kl_loss / kl_loss_type / kl_loss_coef` | True / **low_var_kl(k3)** / **0.01** | `actor/kl_loss`、`actor/kl_coef`、`actor/loss` |
| `rollout.n` | **16** | GRPO 每个 prompt 采 16 条,组内算 advantage |
| `train_batch_size` | 60 | 每 step 60 prompt × 16 = 960 条 rollout |
| `entropy_coeff` | **0** | 所以 log 里**没有** `actor/entropy_loss` 项 |
| `clip_ratio`(默认) | 0.2 | PPO 裁剪,`pg_clipfrac` |
| `optim.lr` | 3e-6 | `actor/lr` |
| `temperature` | 1.0 | 采样温度 |
| `test_freq / total_epochs` | 5 / 15 | 每 5 step 验证一次;共跑到 step 60 |
| 模型 / 硬件 | Qwen2.5-VL-7B-Instruct / 6×4090, FSDP2 | perf/timing 类 |

### 符号与公式约定

- 令 $m_{i,t}$ 是 response mask（掩码），$\ell^\theta_{i,t}$ 是当前 actor logprob，$\ell^{old}_{i,t}$ 是 PPO anchor logprob，$\ell^{roll}_{i,t}$ 是 vLLM rollout logprob。
    - 同一批 rollout 出来的数据 $(x_i, y_i)$，也就是 prompt $x_i$ 和已经生成好的 response token $y_{i,1:L}$，会被不同“版本/引擎”的模型重新打分（score log probability）。
    - 不是重新生成，而是 teacher-forcing 式地算：$\ell_{i,t}=\log \pi(y_{i,t}\mid x_i,y_{i,<t})$
    - $\ell^{roll}_{i,t}=\log \pi_{\text{vLLM rollout}}(y_{i,t}\mid x_i,y_{i,<t})$
    - $\ell^{old}_{i,t}=\log \pi_{\text{actor before PPO update}}(y_{i,t}\mid x_i,y_{i,<t})$
        - 训练 actor 在本 step 更新前，对同一批 rollout response 重新 forward 一遍算出来的 PPO anchor logprob。
        - 如果 bypass_mode=False，所以没有直接把 rollout_log_probs 当 old_log_probs 用。
            - bypass_mode=True, $\ell^{old}_{i,t} \leftarrow \ell^{roll}_{i,t}$，PPO 里的 old policy logprob 不再由训练 actor 重新 forward 计算，而是直接复用 vLLM rollout 时记录的 logprob。
            - $\rho_{i,t}=\exp(\ell^\theta_{i,t}-\ell^{old}_{i,t})=\exp(\ell^\theta_{i,t}-\ell^{roll}_{i,t})=\frac{\pi_\theta(y_{i,t}\mid x_i,y_{i,<t})}{\pi_{roll}(y_{i,t}\mid x_i,y_{i,<t})}$
            - 这个 $\rho_{i,t}\in\mathbb{R}^{B\times L_r}$ 就是 per-token importance ratio（重要性采样比率）。
    - $\ell^{ref}_{i,t}=\log \pi_{\text{ref}}(y_{i,t}\mid x_i,y_{i,<t})$
- masked mean（掩码平均）是

$$
\operatorname{mean}_m(x)=\frac{\sum_{i,t}x_{i,t}m_{i,t}}{\sum_{i,t}m_{i,t}+10^{-8}}.
$$

- PPO ratio（概率比）：$r_{i,t}=\exp(\ell^\theta_{i,t}-\ell^{old}_{i,t})$
- actor/ppo_kl = mean_m($\ell^{old}-\ell^\theta$)
- actor/kl_loss 使用 low-var KL（k3 estimator）相对 reference model：$\delta=\ell^{ref}-\ell^\theta,\quad KL_{k3}=\exp(\delta)-\delta-1$
- actor/entropy 是旧策略 logprob 阶段从 logits 算的 token 熵 $H(p)=-\sum_v p_v\log p_v$（监控和诊断指标），不是 entropy regularizer loss。
- Rollout 一致性（rollout_is）
    - training/rollout_probs_diff_*：比较 $p^\theta=\exp(\ell^{old})$ 与 $p^{roll}=\exp(\ell^{roll})$ 的绝对差
    - pearson_corr 是两组概率的 Pearson correlation（皮尔逊相关）。
- actor/loss = actor/pg_loss + 0.01 * actor/kl_loss，如果 entropy_coeff=0，entropy 不进入 loss。

设有效 token mask 为 $m_{i,t}$，序列级 masked mean 为
$\bar{\ell}^{old}_i=\frac{\sum_t m_{i,t}\ell^{old}_{i,t}}{\sum_t m_{i,t}}$，$\bar{\ell}^{roll}_i=\frac{\sum_t m_{i,t}\ell^{roll}_{i,t}}{\sum_t m_{i,t}}$，全局 masked mean 为 $\operatorname{mean}_m(\cdot)$。
$$
\begin{split}
\text{training\_ppl}&=\frac{1}{B}\sum_i \exp(-\bar{\ell}^{old}_i)\\
\text{rollout\_ppl}&=\frac{1}{B}\sum_i \exp(-\bar{\ell}^{roll}_i)\\
\text{kl}&=\operatorname{mean}_m(\ell^{roll}_{i,t}-\ell^{old}_{i,t})\\
\text{k3\_kl}&=\operatorname{mean}_m\left(\exp(\ell^{old}_{i,t}-\ell^{roll}_{i,t})-(\ell^{old}_{i,t}-\ell^{roll}_{i,t})-1\right)\\
\text{log\_ppl\_diff}&=\frac{1}{B}\sum_i(\bar{\ell}^{roll}_i-\bar{\ell}^{old}_i)\\
\text{ppl\_ratio}&=\frac{1}{B}\sum_i \exp(\bar{\ell}^{roll}_i-\bar{\ell}^{old}_i)\\
\text{chi2\_token}&=\operatorname{mean}_m\left(\exp(2(\ell^{old}_{i,t}-\ell^{roll}_{i,t}))\right)-1\\
\text{chi2\_seq}&=\frac{1}{B}\sum_i \exp\left(2\sum_t m_{i,t}(\ell^{old}_{i,t}-\ell^{roll}_{i,t})\right)-1
\end{split}
$$
其中 $\pi^{old}/\pi^{roll}=\exp(\ell^{old}-\ell^{roll})$。

- 关于 ppl
    - https://huggingface.co/docs/transformers/perplexity
    - $\text{PPL}(X) = \exp\left\{ -\frac{1}{t}\sum_i^t \log p_\theta (x_i|x_{<i}) \right\}$

### critic/*

| 指标 | 公式 | 含义 / 本 run |
|---|---|---|
| `critic/score/{mean,max,min}` | 每条序列 `token_level_scores.sum(-1)` 再对**非 abort 样本**求统计 | 就是上面 reward 的 `score`。mean: step1 **0.297 → step60 0.534**,稳步上升 |
| `critic/rewards/{…}` | 每条 `token_level_rewards.sum(-1)` 的统计 | 因 `use_kl_in_reward=False`,**恒等于 `critic/score`** |
| `critic/advantages/{mean,max,min}` | GRPO 优势(见 §3),在 response mask 上取统计 | mean≈**-0.03≈0**(组内归一后应为 0,健康);max/min 常年 **±3.75** ← 见下方彩蛋 |
| `critic/returns/{…}` | GRPO 里 `returns := advantages` | **恒等于 `critic/advantages`**(无 value function) |

> DAPO / Dr.GRPO / GSPO ....
```
r_i = 序列标量奖励(token_level_rewards.sum)
对每个 prompt 组 G(|G|=16):
    A_i = (r_i − mean_G(r)) / (std_G(r) + 1e-6)     # norm_adv_by_std=True(原版GRPO)
A_i 广播到该序列所有 response token(乘 response_mask)
returns := advantages
```

```
# 关于 token_level_scores 的计算
token_level_scores.shape == (batch_size, response_length) 
    batch_size = train_batch_size × rollout.n = 60 × 16 = 960(每 prompt 采 16 条)
    response_length = padding 到 max_response_length = 1024

# workers/reward_manager/naive.py
# 先全部填 0，形状 = responses
reward_tensor = torch.zeros_like(data.batch["responses"], dtype=torch.float32)

for i in range(len(data)):                                  # 逐条序列
    valid_response_length = attention_mask[prompt_length:].sum()   # 该条真实(非pad)长度
    score = self.compute_score(...)                         # custom reward 函数
    reward = score["score"]                                 # 取出那个标量 0.9·acc+0.1·format
    # 只把标量写到「最后一个有效 token」这一个位置，其余保持 0
    reward_tensor[i, valid_response_length - 1] = reward

# ray_trainer.py L1608: 这个张量就成了 token_level_scores
batch.batch["token_level_scores"] = reward_tensor
```

- 也就是说,每一行 1024 个数里,只有 1 个非零(落在该条回复 EOS/最后一个有效 token 处),其余全是 0:

```
第 i 行:  [0, 0, 0, ..., 0, r_i, 0, 0, ..., 0]
                            ↑ index = valid_response_length_i - 1（每行位置不同）
```

### PPO vs. GRPO

| 维度 | PPO / GAE | GRPO |
|---|---|---|
| `adv_estimator` | `gae` | `grpo` |
| critic | 默认需要 critic | 默认不需要 critic |
| baseline | learned value $V_{\phi}(s_t)$ | 同 prompt 多个 response 的组内均值 |
| advantage 粒度 | token-level，由 TD/GAE 递推 | response-level scalar，广播到所有 response tokens |
| return | $R_t = A_t + V_t$ | $\texttt{returns}=\texttt{advantages}$ |
| rollout 数 | 单样本也合理 | 需要每个 prompt 多采样才有组内相对比较，典型 `rollout.n > 1` |
| actor loss | PPO clip | 仍是 PPO clip，只是 $A$ 的来源变了 |


```
token_level_rewards: [B, Lr]
values:              [B, Lr]
response_mask:       [B, Lr]
```

- 公式是标准 GAE。对 response token 从后往前递推：
    - $\delta_t = r_t + \gamma V_{\text{old}}(s_{t+1}) - V_{\text{old}}(s_t)$
    - $A_t^{GAE} = \sum_{l\ge0}(\gamma\lambda)^l\delta_{t+l}$
```
delta = token_level_rewards[:, t] + gamma * nextvalues - values[:, t]
lastgaelam_ = delta + gamma * lam * lastgaelam
```
- $R_t = A_t + V_t$（$R_t^\lambda = A_t^{GAE} + V_{\text{old}}(s_t)$）
```
returns = advantages + values
advantages = verl_F.masked_whiten(advantages, response_mask)
```
- advantages 参与 PPO pg loss，returns 参与的是 critic 的训练
    - $A_t \approx R_t - V_{\text{old}}(s_t)$
    - critic 要学一个 value function：$V_\phi(s_t) \approx R_t$，value mse loss $L_V(\phi)=\frac12\left(V_\phi(s_t) - R_t\right)^2$

------
$V_t$ 是 baseline，目标是接近“期望 return”，不是接近每条样本的实际 outcome 到完全相等。
- PPO/GAE 里可以粗略看成：$A_t \approx R_t - V_{\text{old}}(s_t)$ 
- 如果 critic 很好，
    - 那么：$V(s_t) \approx \mathbb{E}[R_t \mid s_t]$
    - 所以优势变成：$A_t \approx R_t - \mathbb{E}[R_t \mid s_t]$
- 这不会让每个样本都是 0，只会让同一状态下的 advantage 均值接近 0。高于预期的 action/response 是正 advantage，低于预期的是负 advantage。

### DAPO / Dr.GRPO / GSPO / GMPO / CISPO

- loss agg mode:
    - GRPO length bias

### use removing pad

以一个 micro-batch 为例：
```
原始 padded:
seq0: [p p p r r PAD PAD]  len=5
seq1: [p p r r r r PAD]    len=6
seq2: [p r PAD PAD PAD]    len=2

input_ids:      [3, 7]
attention_mask: [3, 7]
```
remove padding 后变成：

```
values:
[p p p r r | p p r r r r | p r]

offsets / cu_seqlens:
[0, 5, 11, 13]
```

所以每条 seq 的边界是：
```
seq0 = values[0:5]
seq1 = values[5:11]
seq2 = values[11:13]
```

拼接后不同样本会不会互相 attention？不会。边界靠 position_ids/cu_seqlens 传给 varlen attention。

```python
cu_seqlens = [0, len0, len0+len1, ...]
max_seqlen = max(len_i)

flash_attn_varlen_func(
    q, k, v,
    cu_seqlens_q=cu_seqlens,
    cu_seqlens_k=cu_seqlens,
    max_seqlen_q=max_seqlen,
    max_seqlen_k=max_seqlen,
    causal=True,
)
```

### attn implementation

- sdpa
- flash_attention_2

### Importance Sampling

- token-level IS
$$
w^{token}_{i,t}=\exp(\ell^\theta_{i,t}-\ell^{roll}_{i,t}) \quad [B,L_r]
$$
- sequence-level IS
$$
w^{seq}_{i,t}= \exp\left(\sum_t m_{i,t}(\ell^\theta_{i,t}-\ell^{roll}_{i,t})\right) \quad [B,L_r]
$$
- sequence 模式会把同一个序列级权重 broadcast 到该 response 的所有 token。

```python
algorithm.rollout_correction.loss_type=reinforce
algorithm.rollout_correction.rollout_is=token 或 sequence
```

才会显式使用 IS weight：
$$
L^{reinforce}_{i,t}=-w_{i,t}\,\log\pi_\theta(y_{i,t}\mid x_i,y_{i,<t})\,A_{i,t}
$$

bypass_mode=True 默认没有新的额外 IS；它把 PPO ratio 本身改成 $\pi_\theta/\pi_{roll}$，这就是隐式/内置的 IS。只有切到 loss_type="reinforce" 并设置 rollout_is，才会真的额外乘显式 IS 权重。